In [0]:
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]
columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count"
]
df = spark.createDataFrame(data, columns)
df.show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+



In [0]:
from pyspark.sql.functions import col,when
df = df.withColumn("total_bill",col("consultation_fee") + col("tests_count") * 500) \
       .withColumn("category",when(col("total_bill") >= 6000, "High").when(col("total_bill") >= 3000, "Medium")
        .otherwise("Low"))
df.show()

+--------+------------+---------+-----------+----------------+-----------+----------+--------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|total_bill|category|
+--------+------------+---------+-----------+----------------+-----------+----------+--------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|      5500|  Medium|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|      4000|  Medium|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|      2000|     Low|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|      6000|    High|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|      7500|    High|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|      4500|  Medium|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|      5500|  Medium|
|     108|  Meera Iyer|Bangalore|Dermatology|     

In [0]:
high_value_df = df.filter(col("total_bill") >= 6000)
high_value_df.show()

+--------+------------+---------+----------+----------------+-----------+----------+--------+
|visit_id|patient_name|     city|department|consultation_fee|tests_count|total_bill|category|
+--------+------------+---------+----------+----------------+-----------+----------+--------+
|     104|  Priya Nair|Bangalore|Cardiology|            5000|          2|      6000|    High|
|     105|Vikram Singh|  Chennai| Neurology|            7000|          1|      7500|    High|
+--------+------------+---------+----------+----------------+-----------+----------+--------+



In [0]:
from pyspark.sql.functions import count,sum
agg_df = df.groupBy("department") .agg(count("*").alias("total_patients"),sum("total_bill").alias("total_revenue"))
agg_df.show()

+-----------+--------------+-------------+
| department|total_patients|total_revenue|
+-----------+--------------+-------------+
| Cardiology|             3|        17000|
|Orthopedics|             2|         8500|
|Dermatology|             2|         4500|
|  Neurology|             1|         7500|
+-----------+--------------+-------------+



In [0]:
from pyspark.sql.functions import desc
sorted_df = agg_df.orderBy(desc("total_revenue"))
sorted_df.show()

+-----------+--------------+-------------+
| department|total_patients|total_revenue|
+-----------+--------------+-------------+
| Cardiology|             3|        17000|
|Orthopedics|             2|         8500|
|  Neurology|             1|         7500|
|Dermatology|             2|         4500|
+-----------+--------------+-------------+



In [0]:
df.createOrReplaceTempView("patients")

In [0]:
%sql
SELECT * FROM patients WHERE department = 'Cardiology';

visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,category
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500,Medium
104,Priya Nair,Bangalore,Cardiology,5000,2,6000,High
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500,Medium


In [0]:
%sql
SELECT city, SUM(consultation_fee + tests_count * 500) AS revenue FROM patients GROUP BY city;

city,revenue
Hyderabad,5500
Delhi,4000
Mumbai,2000
Bangalore,8500
Chennai,7500
Kolkata,4500
Ahmedabad,5500


In [0]:
%sql
SELECT *, (consultation_fee + tests_count * 500) AS total_bill FROM patients ORDER BY total_bill DESC LIMIT 3;

visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,category,total_bill
105,Vikram Singh,Chennai,Neurology,7000,1,7500,High,7500
104,Priya Nair,Bangalore,Cardiology,5000,2,6000,High,6000
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500,Medium,5500


In [0]:
%sql
SELECT department, COUNT(*) AS total_patients FROM patients GROUP BY department;

department,total_patients
Cardiology,3
Orthopedics,2
Dermatology,2
Neurology,1


In [0]:
df.write.format("delta") .mode("overwrite") .saveAsTable("patient_table")
patient_df = spark.read.table("patient_table")
patient_df.show()

+--------+------------+---------+-----------+----------------+-----------+----------+--------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|total_bill|category|
+--------+------------+---------+-----------+----------------+-----------+----------+--------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|      5500|  Medium|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|      4000|  Medium|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|      2000|     Low|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|      6000|    High|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|      7500|    High|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|      4500|  Medium|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|      5500|  Medium|
|     108|  Meera Iyer|Bangalore|Dermatology|     

In [0]:
new_data = [(110,"Vijay Kumar","Pune","Cardiology",5000,2)]
new_df = spark.createDataFrame(new_data, columns)
new_df.write.format("delta") .mode("append") .saveAsTable("patient_table")

In [0]:
from delta.tables import DeltaTable
patient_table = DeltaTable.forName(spark, "patient_table")
patient_table.update(condition="visit_id = 101",set={"consultation_fee": "7500"})

DataFrame[num_affected_rows: bigint]

In [0]:
patient_table.delete("visit_id = 108")

DataFrame[num_affected_rows: bigint]

In [0]:
updates = [(101,"Arjun Reddy","Hyderabad","Cardiology",6000,2),(110,"New Patient","Chennai","Dermatology",2000,1)]
updates_df = spark.createDataFrame(updates, columns)
updates_df = updates_df.withColumn("total_bill",col("consultation_fee") + col("tests_count") * 500)
updates_df = updates_df.withColumn("category",when(col("total_bill") >= 6000, "High").when(col("total_bill") >= 3000, "Medium").otherwise("Low"))
patient_table.alias("t").merge(updates_df.alias("s"),"t.visit_id = s.visit_id") .whenMatchedUpdateAll() .whenNotMatchedInsertAll() .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
from delta.tables import DeltaTable
patient_table = DeltaTable.forName(spark, "patient_table")
patient_table.history().show()

+-------+-------------------+---------------+--------------------+--------------------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|         userId|            userName|           operation| operationParameters| job|          notebook|queryHistoryStatementId|           clusterId|readVersion|   isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+---------------+--------------------+--------------------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|      7|2026-05-04 05:46:50|141544077535084|azuser6413_mml.lo...|            OPTIMIZE|{clusterBy -> [],...|NULL|{3482506473663282}|   4565a4fa-799e-4f9...|0504-035832-8

In [0]:
df_old = spark.read.format("delta") .option("versionAsOf", 0) .table("patient_table")
df_old.show()

+--------+------------+---------+-----------+----------------+-----------+----------+--------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|total_bill|category|
+--------+------------+---------+-----------+----------------+-----------+----------+--------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|      5500|  Medium|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|      4000|  Medium|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|      2000|     Low|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|      6000|    High|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|      7500|    High|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|      4500|  Medium|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|      5500|  Medium|
|     108|  Meera Iyer|Bangalore|Dermatology|     

In [0]:
%sql
VACUUM patient_table RETAIN 168 HOURS DRY RUN;

path


In [0]:
df.write.mode("overwrite").parquet("/tmp/patient_parquet")

In [0]:
from delta.tables import DeltaTable
DeltaTable.convertToDelta(spark,"parquet.`/tmp/patient_parquet`")

In [0]:
delta_df = spark.read.format("delta").load("/tmp/patient_parquet")
delta_df.show()

+--------+------------+---------+-----------+----------------+-----------+----------+--------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|total_bill|category|
+--------+------------+---------+-----------+----------------+-----------+----------+--------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|      5500|  Medium|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|      5500|  Medium|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|      4000|  Medium|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|      4500|  Medium|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|      6000|    High|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|      2500|     Low|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|      2000|     Low|
|     105|Vikram Singh|  Chennai|  Neurology|     

In [0]:
df.write.format("delta") .mode("overwrite") .saveAsTable("patient_target")

In [0]:
updates = [
(101,"Arjun Reddy","Hyderabad","Cardiology",6000,2),
(111,"Preethy Mishra","Chennai","Dermatology",2000,1)  
]

updates_df = spark.createDataFrame(updates, columns)

In [0]:
from pyspark.sql.functions import col, when
updates_df = updates_df.withColumn("total_bill",col("consultation_fee") + col("tests_count") * 500)
updates_df = updates_df.withColumn("category",when(col("total_bill") >= 6000, "High").when(col("total_bill") >= 3000, "Medium").otherwise("Low"))

In [0]:
from delta.tables import DeltaTable
target_table = DeltaTable.forName(spark, "patient_target")
target_table.alias("t").merge(updates_df.alias("s"),"t.visit_id = s.visit_id") .whenMatchedUpdateAll() .whenNotMatchedInsertAll() .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
CREATE CATALOG hospital_catalog;

In [0]:
%sql
CREATE SCHEMA hospital_catalog.patient_schema;

In [0]:
%sql
CREATE TABLE hospital_catalog.patient_schema.patients
USING DELTA
AS SELECT * FROM patient_table;

In [0]:
spark.table("silver_patients") .write.format("delta") .mode("overwrite") .saveAsTable("hospital_catalog.patient_schema.patients")

In [0]:
%sql
SHOW TABLES IN hospital_catalog.patient_schema;

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
SHOW SCHEMAS IN hospital_catalog;

In [0]:
%sql
CREATE TABLE hospital_catalog.patient_schema.high_value_patients AS
SELECT *
FROM hospital_catalog.patient_schema.patients
WHERE category = 'High';

In [0]:
%sql
GRANT SELECT ON TABLE hospital_catalog.patient_schema.patients
TO `data_analyst`;

In [0]:
%sql
SELECT *
FROM system.access.audit_logs
WHERE service_name = 'unityCatalog'
ORDER BY event_time DESC;